**数据预处理：pandas**<a id='toc0_'></a>    
- [读取数据集](#toc1_)    
- [处理缺失值](#toc2_)    
- [转换为张量格式](#toc3_)    
- [小结](#toc4_)    
- [练习](#toc5_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<div style="font-size:32px;  padding:5px;"> 
数据预处理：pandas
</div>

:label:`sec_pandas`

为了能用深度学习来解决现实世界的问题，我们经常从预处理原始数据开始，
而不是从那些准备好的张量格式数据开始。
在Python中常用的数据分析工具中，我们通常使用`pandas`软件包。
像庞大的Python生态系统中的许多其他扩展包一样，`pandas`可以与张量兼容。
本节我们将简要介绍使用`pandas`预处理原始数据，并将原始数据转换为张量格式的步骤。
后面的章节将介绍更多的数据预处理技术。

⚠️ `pandas`中`fillna()`等数据处理方法默认是非原地（non-inplace）的

$\implies$ 它们不会直接修改原始`DataFrame/Series`，而是返回一个修改后的新副本。

# <a id='toc1_'></a>[读取数据集](#toc0_)

举一个例子，我们首先(**创建一个人工数据集，并存储在CSV（逗号分隔值）文件**)
`../data/house_tiny.csv`中。
以其他格式存储的数据也可以通过类似的方式进行处理。
下面我们将数据集按行写入CSV文件中。

In [ ]:
import os
# 手搓数据集
os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

要[**从创建的CSV文件中加载原始数据集**]，我们导入`pandas`包并调用`read_csv`函数。该数据集有四行三列。其中每行描述了房间数量（“NumRooms”）、巷子类型（“Alley”）和房屋价格（“Price”）。


In [1]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import os
import pandas as pd
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


# <a id='toc2_'></a>[处理缺失值](#toc0_)

⚠️ “NaN”项代表缺失值。$\impliedby$ 处理的典型方法包括：

* 【**插值法**】：用一个替代值弥补缺失值。
* 【**删除法**】：直接忽略缺失值。



(这里，我们将考虑【**插值法**】)。

通过位置索引`iloc`，我们将`data`分成`inputs`和`outputs`，
其中前者为`data`的前两列，而后者为`data`的最后一列。
对于`inputs`中缺少的数值，我们用同一列的均值替换“NaN”项。


In [2]:
data.iloc[:, 0] = data.iloc[:,0].fillna(data.iloc[:,0].mean())# 需要在下一句之前
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]# 分开并起名
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


⚠️  `pandas`使用`[]`会被误解成按列名筛选，因此需要使用位置索引`iloc`或者标签索引`loc`来选择

In [ ]:
inputs['NumRooms'] = inputs['NumRooms'].fillna(inputs['NumRooms'].mean())# 第三种办法：直接使用列名
print(inputs)

[**对于`inputs`中的类别值或离散值，我们将“NaN”视为一个类别。**]
由于“巷子类型”（“Alley”）列只接受两种类型的类别值“Pave”和“NaN”，
`pandas`可以自动将此列转换为两列“Alley_Pave”和“Alley_nan”。
巷子类型为“Pave”的行会将“Alley_Pave”的值设置为1，“Alley_nan”的值设置为0。
缺少巷子类型的行会将“Alley_Pave”和“Alley_nan”分别设置为0和1。


In [ ]:
#import numpy as np
#inputs['Alley'] = inputs['Alley'].replace('NaN', np.nan)
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)#不设置为int就出现的是false或者true
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


# <a id='toc3_'></a>[转换为张量格式](#toc0_)

现在`inputs`和`outputs`中的所有条目都是【**数值类型** 】

$\implies$ 可以转换为【**张量格式**】。

当数据采用张量格式后，可以通过在[ndarray](ndarray.ipynb)中引入的那些张量函数来进一步操作。


In [6]:
import torch

X1 = torch.tensor(inputs.to_numpy(dtype=float))
y1 = torch.tensor(outputs.to_numpy(dtype=float))
X1, y1

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

In [7]:
X2,y2 = torch.tensor(inputs.values),torch.tensor(outputs.values)
X2,y2

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500, 106000, 178100, 140000]))

# <a id='toc4_'></a>[小结](#toc0_)

* `pandas`软件包是Python中常用的数据分析工具中，`pandas`可以与张量兼容。
* 用`pandas`处理缺失的数据时，我们可根据情况选择用插值法和删除法。

# <a id='toc5_'></a>[练习](#toc0_)

创建包含更多行和列的原始数据集。

1. 删除缺失值最多的列。
2. 将预处理后的数据集转换为张量格式。


In [8]:
import os
import pandas as pd
data_file = os.path.join('..', 'data', 'house_tiny.csv')
data = pd.read_csv(data_file)


In [11]:
# 删除缺失值最多的列
null_count = data.iloc[:,0:2].isnull().sum()
max_null = null_count.idxmax()
data.drop(max_null, axis = 1, inplace=True)
print(data)

   NumRooms   Price
0       NaN  127500
1       2.0  106000
2       4.0  178100
3       NaN  140000


In [13]:
data_tensor = torch.tensor(data.values)
print(data_tensor)

tensor([[       nan, 1.2750e+05],
        [2.0000e+00, 1.0600e+05],
        [4.0000e+00, 1.7810e+05],
        [       nan, 1.4000e+05]], dtype=torch.float64)
